# Convert and Run Polyglot-Lion with OpenVINO™

[Polyglot-Lion 1.7B](https://huggingface.co/knoveleng/polyglot-lion-1.7b) is a
multilingual speech-recognition model based on [Qwen3-ASR](https://github.com/QwenLM/Qwen3-ASR).
This tutorial runs the original PyTorch model on CPU, converts it to stateful
OpenVINO IR, compares recognition accuracy on the same audio, and benchmarks
inference on Intel CPU and optional GPU. Optional weight compression demonstrates
the trade-off between model size, accuracy and generation latency.

**Experimental notebook:** this example uses a pinned custom Optimum Intel
revision and explicit encoder/precision corrections. The default one-clip
smoke test does not certify multilingual deployment. See [Conclusions](#Conclusions)
and the [README](README.md) for validation results and known limitations.

<div class="alert alert-block alert-danger"><b>Important note:</b>
This notebook is validated on Linux with Python 3.12. Other platforms and
Python versions are not certified.</div>

#### Table of contents:

- [Prerequisites](#Prerequisites)
    - [Install dependencies](#Install-dependencies)
- [Configure the tutorial](#Configure-the-tutorial)
    - [Select inference devices](#Select-inference-devices)
- [Prepare audio samples](#Prepare-audio-samples)
- [Load the PyTorch model](#Load-the-PyTorch-model)
- [Convert model to OpenVINO IR](#Convert-model-to-OpenVINO-IR)
- [Compress model weights](#Compress-model-weights)
    - [Full INT8 weight compression](#Full-INT8-weight-compression)
    - [Mixed INT8/FP16 weight compression](#Mixed-INT8/FP16-weight-compression)
- [Validate OpenVINO model accuracy](#Validate-OpenVINO-model-accuracy)
- [Benchmark PyTorch and OpenVINO](#Benchmark-PyTorch-and-OpenVINO)
- [Interactive demo](#Interactive-demo)
- [Conclusions](#Conclusions)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/polyglot-lion/polyglot-lion.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Use a dedicated Python 3.12 kernel with pip and Git. PyTorch runs on CPU;
an XPU-enabled PyTorch wheel is not needed for OpenVINO GPU inference.
Allow about 16 GB for the checkpoint and all optional model variants, plus
the Python environment and download caches. The initial conversion can take
more than five minutes on slower machines.

Keep [polyglot_helper.py](polyglot_helper.py) beside the notebook.
The next cell copies the standard utilities from the checkout, or downloads
a pinned copy when the notebook is run outside the repository.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import shutil
import urllib.request

if not Path("polyglot_helper.py").is_file():
    raise RuntimeError("Keep polyglot_helper.py beside this notebook and start the kernel in that folder.")
utils_source = Path("../../utils/notebook_utils.py")
if utils_source.is_file():
    shutil.copyfile(utils_source, "notebook_utils.py")
elif not Path("notebook_utils.py").is_file():
    url = "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/5f6a94b9863affdef9d7d4b7905204d0cf47fdee/utils/notebook_utils.py"
    with urllib.request.urlopen(url, timeout=30) as response:
        Path("notebook_utils.py").write_bytes(response.read())

### Install dependencies
[back to top ⬆️](#Table-of-contents:)

Set `INSTALL_PACKAGES=True` for the first run. After installation, restart
the kernel, set it back to `False`, and run the notebook from the beginning.
The pinned custom Optimum Intel revision supplies Qwen3-ASR export support.
These versions target the original `qwen-asr` implementation; do not replace
them with the Hugging Face-native Qwen3-ASR versions used in other tutorials.

In [ ]:
DEPENDENCIES = [
    "torch==2.15.0+cpu",
    "torchaudio==2.11.1+cpu",
    "openvino==2026.4.1",
    "nncf==3.4.0",
    "qwen-asr==0.0.6",
    "transformers==4.57.6",
    "datasets==4.8.5",
    "numpy>=1.26,<3",
    "librosa",
    "soundfile",
    "pandas",
    "ipywidgets",
    "huggingface-hub",
    "optimum-intel @ git+https://github.com/openvino-dev-samples/optimum-intel.git@4ca1144eafc3ef7d3d805a99c7b92953441437e5",
]
INSTALL_PACKAGES = False
if INSTALL_PACKAGES:
    if importlib.util.find_spec("pip") is None:
        raise RuntimeError("Select a kernel with pip, e.g. create it using uv venv --seed --python 3.12.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--extra-index-url", "https://download.pytorch.org/whl/cpu", *DEPENDENCIES], check=True)
    raise RuntimeError("Installation complete: restart the kernel and disable INSTALL_PACKAGES.")

Enable the standard execution telemetry. To disable it,
set `SCARF_NO_ANALYTICS=1` before launching Jupyter.

In [ ]:
# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("polyglot-lion.ipynb")

## Configure the tutorial
[back to top ⬆️](#Table-of-contents:)

Import the inference and evaluation utilities. The helper contains only
model-specific runtime, data and metric code; conversion and evaluation
remain visible in the notebook.

The defaults run corrected FP16 inference on CPU with one English clip.
Optional steps are controlled here, before model loading or conversion:

| Option | Purpose |
|---|---|
| `RUN_GPU` | Also evaluate an available Intel GPU. |
| `RUN_MIXED_INT8` | Build the mixed INT8/FP16 weight candidate. |
| `RUN_INT8` | Build experimental full INT8 weights; known accuracy failures remain. |
| `RUN_MULTILINGUAL_EVALUATION` | Evaluate English, Mandarin, Tamil and Malay. |
| `MULTILINGUAL_SAMPLES_PER_LANGUAGE` | Set the size of that optional first-N slice. |
| `RUN_F32_DIAGNOSTICS` | Load a separate global-f32 model for logit diagnostics. |

Keep `GPU_DECODER_F32=True`: the corrected encoder alone does not resolve
the known Tamil 1666 failure with default GPU decoder f16 arithmetic.
The decoder requests f32 explicitly; the encoder keeps its task settings.
Weight storage precision is not a guarantee of inference arithmetic.

In [ ]:
import json
from importlib.metadata import version

import nncf
import openvino as ov
import pandas as pd
import torch
import ipywidgets as widgets
from IPython.display import Audio, display
from huggingface_hub import snapshot_download
from optimum.intel import OVModelForSpeechSeq2Seq
from qwen_asr.core.transformers_backend.modeling_qwen3_asr import Qwen3ASRForConditionalGeneration
from transformers import AutoProcessor
from notebook_utils import device_widget
from polyglot_helper import (
    MODEL_ID,
    MODEL_REVISION,
    DATASET_ID,
    DATASET_REVISION,
    LANGUAGES,
    AUDIO_ENCODER_FIX,
    export_audio_encoder,
    configure_audio_encoder,
    benchmark,
    corpus_error,
    generate,
    load_samples,
    logit_metrics,
    model_fingerprint,
    prepare_inputs,
    transcribe,
    restore_decoder_projections,
)

RUN_GPU = False
RUN_INT8 = False
RUN_MIXED_INT8 = False
GPU_DECODER_F32 = True
RUN_MULTILINGUAL_EVALUATION = False
MULTILINGUAL_SAMPLES_PER_LANGUAGE = 64
RUN_F32_DIAGNOSTICS = False
MAX_NEW_TOKENS = 512
ERROR_BUDGET = 0.02
CONFIG = {"PERFORMANCE_HINT": "LATENCY", "CACHE_DIR": ""}
COMPONENT_CONFIGS = {"CPU": {}, "GPU": {"decoder": {"INFERENCE_PRECISION_HINT": "f32"}} if GPU_DECODER_F32 else {}}
MODEL_DIR, OUTPUT_DIR = Path("model"), Path("output")
FP16, INT8 = MODEL_DIR / "fp16-full-context", MODEL_DIR / "int8-full-context"
MIXED = MODEL_DIR / "mixed-int8-first8-full-context"

### Select inference devices
[back to top ⬆️](#Table-of-contents:)

CPU is always evaluated. If `RUN_GPU=True`, add an available Intel GPU;
otherwise report that GPU evaluation was skipped. Display package versions
and component compile settings so the measured configuration is explicit.

In [ ]:
core = ov.Core()
devices = ["CPU"]
if RUN_GPU:
    if "GPU" in {device.split(".")[0] for device in core.available_devices}:
        devices.append("GPU")
    else:
        print("GPU explicitly skipped: no Intel GPU is available. CPU results are not GPU approval.")
versions = {name: version(name) for name in ("torch", "torchaudio", "openvino", "nncf", "transformers", "optimum-intel", "qwen-asr")}
if versions["qwen-asr"] != "0.0.6":
    raise RuntimeError("The encoder correction targets qwen-asr==0.0.6; install the pinned dependencies.")
print("Versions:", versions)
print("Devices tested:", devices)
print("Explicit component compile settings:", COMPONENT_CONFIGS)
print("REPORT ONLY: no deployment candidate is approved.")
if RUN_GPU and not GPU_DECODER_F32:
    print("EXPERIMENTAL GPU f16 decoder: Tamil 1666 remains a known failure even with FP16 weights.")
if RUN_INT8:
    print("EXPERIMENTAL INT8: encoder correction alone did not resolve the older Tamil 1666 GPU failure.")
if RUN_MIXED_INT8:
    print("Mixed INT8/FP16: FP16 encoder, original attention/first-eight-layer weights, compressed remaining decoder weights.")
    print("GPU evidence uses decoder-only f32; this is weight compression, not INT8 execution or deployment approval.")

The loader below configures the corrected audio encoder and applies any
decoder-only precision override **before compilation**. The same loader is
used for accuracy checks, benchmarks and the interactive demo.

In [ ]:
def load_task_model(folder: Path, device: str, ov_config: dict, component_configs: dict = COMPONENT_CONFIGS):
    model = OVModelForSpeechSeq2Seq.from_pretrained(folder, device=device, trust_remote_code=True, ov_config=ov_config, compile=False)
    configure_audio_encoder(model)
    for component, override in component_configs[device].items():
        part = getattr(model, component)
        part.ov_config = {**part.ov_config, **override}
    model.compile()
    return model

## Prepare audio samples
[back to top ⬆️](#Table-of-contents:)

Download a deterministic test slice of [FLEURS](https://huggingface.co/datasets/google/fleurs)
and prepare mono 16 kHz audio. The default is one English recording.
The model/data revisions are pinned and cached audio hashes are verified.

The optional multilingual first-N slice is not the independent acceptance
set used in the historical results below, and is not speaker-disjoint.
FLEURS is CC-BY-4.0, Google / Conneau et al. (2022),
[paper](https://arxiv.org/abs/2205.12446). Polyglot-Lion is MIT and Qwen3-ASR
is Apache-2.0; see the [source and license information](README.md#sources-and-licenses).

In [ ]:
configs = tuple(LANGUAGES) if RUN_MULTILINGUAL_EVALUATION else ("en_us",)
sample_count = MULTILINGUAL_SAMPLES_PER_LANGUAGE if RUN_MULTILINGUAL_EVALUATION else 1
DATA_DIR = Path("data") / ("multilingual" if RUN_MULTILINGUAL_EVALUATION else "smoke")
samples = load_samples(configs=configs, count=sample_count, directory=DATA_DIR)
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True)
prepared = [prepare_inputs(processor=processor, audio_path=DATA_DIR / sample["audio"]) for sample in samples]
display(Audio(filename=str(DATA_DIR / samples[0]["audio"])))
print("Reference:", samples[0]["reference"])

## Load the PyTorch model
[back to top ⬆️](#Table-of-contents:)

Load the original checkpoint on CPU and transcribe the prepared audio.
PyTorch and OpenVINO use the same inputs, greedy decoding and 512-token cap.
Display the transcript and whether generation reached end-of-sequence (EOS).

In [ ]:
pt_model = Qwen3ASRForConditionalGeneration.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True).eval()
pt_results = [transcribe(pt_model, processor=processor, inputs=inputs, backend="pytorch", max_new_tokens=MAX_NEW_TOKENS) for inputs, _ in prepared]
print("PyTorch:", pt_results[0]["transcription"])
print("EOS completion:", pt_results[0]["complete"])

Prepare one identical teacher-forced token prefix for the later raw-logit
comparison. This numerical diagnostic is separate from task-level WER/CER.

In [ ]:
# One identical teacher-forced prefix for raw-logit diagnostics.
logit_inputs = dict(prepared[0][0])
prefix = pt_results[0]["token_ids"][:8]
logit_inputs["input_ids"] = torch.cat([logit_inputs["input_ids"], torch.tensor([prefix], dtype=torch.long)], dim=1)
logit_inputs["attention_mask"] = torch.ones_like(logit_inputs["input_ids"])
with torch.inference_mode():
    pt_logits = pt_model.thinker(**logit_inputs, use_cache=False).logits[:, -1, :].float().numpy()

## Convert model to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

Optimum Intel exports an audio encoder and a stateful text decoder with
`automatic-speech-recognition-with-past`. Its generation adapter resets
the decoder's KV-cache state between independent utterances.

**Encoder correction:** the pinned original eager/SDPA implementation uses
full-utterance attention after removing CNN padding. The initial exporter
instead used eight-second attention windows and included padded CNN frames.
Re-export the original audio tower with matching semantics, without changing
decoder weights or decoding settings. This does not claim equivalence to
Flash Attention's window semantics; full-context attention has quadratic
memory cost for long audio.

Export once, then verify the source and artifact fingerprints on subsequent
runs. Incomplete or changed exports fail explicitly rather than being
overwritten. Select a new output directory when rebuilding.

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
export_source = {
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "optimum_intel": versions["optimum-intel"],
    "openvino": versions["openvino"],
    "qwen_asr": versions["qwen-asr"],
    "audio_encoder_fix": AUDIO_ENCODER_FIX,
    "reference_encoder_attention": pt_model.thinker.audio_tower.config._attn_implementation,
}
if not FP16.exists():
    checkpoint = snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION)
    exporter = shutil.which("optimum-cli", path=str(Path(sys.executable).parent))
    if exporter is None:
        raise FileNotFoundError("optimum-cli is missing from this kernel environment")
    subprocess.run(
        [
            exporter,
            "export",
            "openvino",
            "--model",
            checkpoint,
            "--task",
            "automatic-speech-recognition-with-past",
            "--trust-remote-code",
            "--weight-format",
            "fp16",
            str(FP16),
        ],
        check=True,
    )
    export_audio_encoder(pt_model.thinker.audio_tower, FP16 / "openvino_encoder_model.xml")
    (FP16 / "export.json").write_text(json.dumps({"source": export_source, "files": model_fingerprint(FP16)}, indent=2))
fp16_record = json.loads((FP16 / "export.json").read_text())
if fp16_record["source"] != export_source or fp16_record["files"] != model_fingerprint(FP16):
    raise ValueError("FP16 export changed; select a new output directory and export explicitly")

variants = {"FP16": FP16}

## Compress model weights
[back to top ⬆️](#Table-of-contents:)

NNCF `INT8_ASYM` compresses eligible weights without calibration data. This
is weight-only compression, not activation PTQ or all-INT8 execution.
Both optional variants are disabled by default.

### Full INT8 weight compression
[back to top ⬆️](#Table-of-contents:)

`RUN_INT8=True` compresses both encoder and decoder weights. This remains an
experimental comparison: decoder-only f32 fixes the three known Tamil
controls, but full INT8 failed a broader Mandarin gate (+2.58 percentage
points). Global f32 is not a substitute; it regresses Tamil 1916 with
compressed weights.

In [ ]:
if RUN_INT8:
    if not INT8.exists():
        INT8.mkdir()
        for name in ("config.json", "generation_config.json"):
            shutil.copyfile(FP16 / name, INT8 / name)
        for component in ("encoder", "decoder"):
            filename = f"openvino_{component}_model.xml"
            compressed = nncf.compress_weights(core.read_model(FP16 / filename), mode=nncf.CompressWeightsMode.INT8_ASYM)
            ov.save_model(compressed, INT8 / filename, compress_to_fp16=False)
            del compressed
        (INT8 / "compression.json").write_text(
            json.dumps({"source": model_fingerprint(FP16), "nncf": versions["nncf"], "mode": "INT8_ASYM", "files": model_fingerprint(INT8)}, indent=2)
        )
    compression = json.loads((INT8 / "compression.json").read_text())
    if compression["source"] != model_fingerprint(FP16) or compression["nncf"] != versions["nncf"] or compression["files"] != model_fingerprint(INT8):
        raise ValueError("INT8 source/software/artifact changed; rebuild explicitly in a new directory")
    variants["INT8 weights"] = INT8

### Mixed INT8/FP16 weight compression
[back to top ⬆️](#Table-of-contents:)

`RUN_MIXED_INT8=True` keeps the corrected encoder FP16, compresses decoder
weights and restores 136 original FP16 projection inputs: attention in all
layers and all projections in layers 0-7. Remaining eligible decoder weights
stay compressed. GPU inference uses the same decoder-only f32 policy.

This candidate passed the local acceptance gates described in
[Conclusions](#Conclusions). It reduced BIN storage by 26.10%, but did not
demonstrate a latency speedup. It is not full INT8 or deployment approval.

In [ ]:
if RUN_MIXED_INT8:
    mixed_source = {
        "fp16": model_fingerprint(FP16),
        "nncf": versions["nncf"],
        "policy": "attention-and-first8-fp16-v1",
        "expected_restored_projections": 136,
    }
    if not MIXED.exists():
        MIXED.mkdir()
        for name in ("config.json", "generation_config.json", "openvino_encoder_model.xml", "openvino_encoder_model.bin"):
            shutil.copyfile(FP16 / name, MIXED / name)
        filename = "openvino_decoder_model.xml"
        original_decoder = core.read_model(FP16 / filename)
        compressed_decoder = nncf.compress_weights(original_decoder.clone(), mode=nncf.CompressWeightsMode.INT8_ASYM)
        restored = restore_decoder_projections(compressed=compressed_decoder, reference=original_decoder)
        if len(restored) != mixed_source["expected_restored_projections"]:
            raise ValueError(f"Expected 136 restored projections, received {len(restored)}")
        ov.save_model(compressed_decoder, MIXED / filename, compress_to_fp16=False)
        del compressed_decoder, original_decoder
        (MIXED / "compression.json").write_text(
            json.dumps({"source": mixed_source, "restored_projections": restored, "files": model_fingerprint(MIXED)}, indent=2)
        )
    mixed_record = json.loads((MIXED / "compression.json").read_text())
    if mixed_record["source"] != mixed_source or mixed_record["files"] != model_fingerprint(MIXED):
        raise ValueError("Mixed model source/software/artifact changed; rebuild explicitly in a new directory")
    variants["Mixed INT8/FP16"] = MIXED

Compare the combined encoder and decoder BIN sizes for the enabled variants.
File size alone does not establish a compute-precision or performance benefit.

In [ ]:
sizes = {label: sum((folder / f"openvino_{part}_model.bin").stat().st_size for part in ("encoder", "decoder")) for label, folder in variants.items()}
display(pd.DataFrame([{"variant": label, "BIN MiB": size / 2**20} for label, size in sizes.items()]))

## Validate OpenVINO model accuracy
[back to top ⬆️](#Table-of-contents:)

Run the enabled variants on the same prepared inputs as PyTorch.
Compare raw logits using the identical teacher-forced prefix: shape/finite
checks, cosine similarity, RMSE, maximum absolute error and top-1 agreement
are diagnostics, not an `allclose` numerical certificate.

`RUN_F32_DIAGNOSTICS=True` loads a **separate** diagnostic model with a
global f32 hint; it does not change task evaluation or benchmark settings.

In [ ]:
results, numerical_rows = {}, []
for label, folder in variants.items():
    for device in devices:
        candidate = f"{label} {device}"
        model = load_task_model(folder, device=device, ov_config=CONFIG)
        results[candidate] = [
            transcribe(model, processor=processor, inputs=inputs, backend="openvino", max_new_tokens=MAX_NEW_TOKENS) for inputs, _ in prepared
        ]
        diagnostic_model = model
        diagnostic_config = dict(CONFIG)
        if RUN_F32_DIAGNOSTICS:
            diagnostic_config["INFERENCE_PRECISION_HINT"] = "f32"
            diagnostic_model = load_task_model(folder, device=device, ov_config=diagnostic_config)
        output = generate(diagnostic_model, inputs=logit_inputs, backend="openvino", max_new_tokens=1, return_dict_in_generate=True, output_logits=True)
        numerical_rows.append(
            {
                "candidate": candidate,
                "diagnostic_config": diagnostic_config,
                "component_config": COMPONENT_CONFIGS[device],
                **logit_metrics(expected=pt_logits, actual=output.logits[0].float().numpy()),
            }
        )
        del diagnostic_model, model

Compare corpus WER for English, Tamil and Malay, and CER for Mandarin.
Scoring preserves Unicode combining marks and weights edit counts by
reference length. A local slice passes only if all outputs reach EOS and
each language stays within +2 percentage points of both PyTorch and the
same-device FP16 baseline. A smoke-test pass is not deployment approval.

In [ ]:
accuracy_rows = []
for candidate, outputs in results.items():
    device = candidate.rsplit(" ", 1)[1]
    for config in configs:
        indices = [i for i, sample in enumerate(samples) if sample["config"] == config]
        references = [samples[i]["reference"] for i in indices]
        characters = config == "cmn_hans_cn"
        pt_rate = corpus_error(references, [pt_results[i]["transcription"] for i in indices], characters)
        ov_rate = corpus_error(references, [outputs[i]["transcription"] for i in indices], characters)
        fp16_rate = corpus_error(references, [results[f"FP16 {device}"][i]["transcription"] for i in indices], characters)
        complete = all(outputs[i]["complete"] and pt_results[i]["complete"] for i in indices)
        accuracy_rows.append(
            {
                "candidate": candidate,
                "language": LANGUAGES[config],
                "metric": "CER" if characters else "WER",
                "PT %": pt_rate * 100,
                "OV %": ov_rate * 100,
                "delta PT pp": (ov_rate - pt_rate) * 100,
                "delta FP16 pp": (ov_rate - fp16_rate) * 100,
                "complete": complete,
                "local_slice_pass": complete and ov_rate <= pt_rate + ERROR_BUDGET + 1e-12 and ov_rate <= fp16_rate + ERROR_BUDGET + 1e-12,
            }
        )
display(pd.DataFrame(accuracy_rows))
display(pd.DataFrame(numerical_rows))
print("LOCAL SLICE ONLY: this smoke result does not establish broad deployment approval.")

Save transcripts, generated tokens (including failures), sample provenance,
model fingerprints and compile settings to `output/evaluation.json`.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "evaluation.json").write_text(
    json.dumps(
        {
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "dataset": DATASET_ID,
            "dataset_revision": DATASET_REVISION,
            "versions": versions,
            "compile_config": CONFIG,
            "component_compile_config": COMPONENT_CONFIGS,
            "audio_encoder_fix": AUDIO_ENCODER_FIX,
            "max_new_tokens": MAX_NEW_TOKENS,
            "samples": samples,
            "fingerprints": {label: model_fingerprint(folder) for label, folder in variants.items()},
            "pytorch": pt_results,
            "openvino": results,
            "accuracy": accuracy_rows,
            "numerical": numerical_rows,
            "deployment_approved": False,
            "known_blocked_configuration": "GPU default f16 decoder arithmetic; earlier Tamil 1666 failure persists",
        },
        ensure_ascii=False,
        indent=2,
    )
)

## Benchmark PyTorch and OpenVINO
[back to top ⬆️](#Table-of-contents:)

Measure complete generation on the first clip using one warmup and three
timed repetitions. This includes the encoder, decoder prefill, token decoding
and Python generation orchestration. It excludes model loading/compilation,
audio preprocessing and detokenization.

Generated lengths can affect comparisons. These timings are not asynchronous
service throughput or a static/dynamic kernel benchmark. Avoid competing
measurements on the same host.

In [ ]:
first_inputs = prepared[0][0]
timings = [
    {
        "candidate": "PyTorch CPU",
        **benchmark(lambda model=pt_model: transcribe(model, processor=processor, inputs=first_inputs, backend="pytorch", max_new_tokens=MAX_NEW_TOKENS)),
    }
]
del pt_model
for label, folder in variants.items():
    for device in devices:
        model = load_task_model(folder, device=device, ov_config=CONFIG)
        timings.append(
            {
                "candidate": f"{label} {device}",
                **benchmark(lambda model=model: transcribe(model, processor=processor, inputs=first_inputs, backend="openvino", max_new_tokens=MAX_NEW_TOKENS)),
            }
        )
        del model
display(pd.DataFrame(timings))
(OUTPUT_DIR / "performance.json").write_text(
    json.dumps(
        {
            "versions": versions,
            "compile_config": CONFIG,
            "component_compile_config": COMPONENT_CONFIGS,
            "audio_encoder_fix": AUDIO_ENCODER_FIX,
            "fingerprints": {label: model_fingerprint(folder) for label, folder in variants.items()},
            "warmup": 1,
            "iterations": 3,
            "sample": samples[0],
            "results": timings,
        },
        indent=2,
    )
)

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Select an enabled weight variant and device, then use the prepared sample or
enter a local WAV path. Audio is resampled/mixed to mono 16 kHz. Acknowledge
the experimental accuracy limitations before clicking **Transcribe**.
A successful demo does not approve the model for deployment.

In [ ]:
variant_widget = widgets.Dropdown(options=list(variants), description="Weights:")
device = device_widget(default="CPU", exclude=[name for name in core.available_devices + ["AUTO"] if name not in devices])
audio_widget = widgets.Text(value="", description="Local WAV:", placeholder="Optional audio path")
acknowledge_widget = widgets.Checkbox(value=False, description="I accept experimental accuracy limitations")
button, demo_output = widgets.Button(description="Transcribe"), widgets.Output()


def transcribe_selection(_=None):
    if not acknowledge_widget.value:
        raise ValueError("Acknowledge the accuracy limitations before experimental inference")
    path = Path(audio_widget.value).expanduser() if audio_widget.value.strip() else DATA_DIR / samples[0]["audio"]
    inputs, _ = prepare_inputs(processor=processor, audio_path=path)
    model = load_task_model(variants[variant_widget.value], device=device.value, ov_config=CONFIG)
    result = transcribe(model, processor=processor, inputs=inputs, backend="openvino", max_new_tokens=MAX_NEW_TOKENS)
    if not result["complete"]:
        raise RuntimeError("Experimental transcription did not reach EOS within the token budget")
    return result


def display_transcription(_):
    demo_output.clear_output(wait=False)
    result = transcribe_selection()
    with demo_output:
        print("EXPERIMENTAL ONLY:", result["transcription"])


button.on_click(display_transcription)
display(variant_widget, device, audio_widget, acknowledge_widget, button, demo_output)
print("No automatic deployment demo approval. Enable the checkbox explicitly to explore.")

## Conclusions
[back to top ⬆️](#Table-of-contents:)

The following are **historical one-host research measurements**, not recomputed by the
default smoke run. This prior 256-clip set has 64 recordings per language.
Both GPU configurations use the corrected full-context FP16 encoder and
explicitly request **decoder-only f32**. Lower is better.

| Configuration | English WER | Mandarin CER | Tamil WER | Malay WER |
|---|---:|---:|---:|---:|
| Original PyTorch | 5.62% | 5.72% | 37.56% | 11.68% |
| Corrected FP16 GPU | 5.69% | 5.76% | 37.65% | 11.45% |
| Mixed INT8/FP16 GPU | 5.47% | 5.76% | 38.21% | 11.68% |

Both pass the unchanged +2 percentage-point per-language error budget and
EOS requirements against original PyTorch; the mixed candidate also passes
against corrected FP16. Both match all three known Tamil controls exactly:
1666 (181 tokens), 1779 (132 tokens) and 1916 (225 tokens).
This prior set is revalidation data, not untouched acceptance.

Corrected FP16 GPU also matched 40/40 original token sequences on its reserved
slice (Tamil WER 37.42%). The mixed candidate passed the observed slices and
a different newly reserved 40-clip set, with Tamil WER **30.52% for both
PyTorch and mixed weights**. On that new set it matched 35/40 original token
sequences, including 7/10 Tamil: equal WER does not imply identical transcripts.
Language/ID pairs and audio hashes were excluded across slices, not speakers
or globally translated sentence content.

**Do not remove the remedies:** original windowed/padded encoder exports
regressed Tamil, and corrected FP16 with default GPU decoder f16 still
repeated on Tamil 1666. Full INT8 with decoder-only f32 fixed those three
controls but failed a broader Mandarin gate (+2.58pp); it remains experimental.
Neither a passing smoke test nor EOS alone proves multilingual robustness.

Enable `RUN_MIXED_INT8` to reproduce the optional mixed candidate: all attention
and first-eight-layer projection inputs stay FP16, while remaining eligible
decoder weights are compressed. Its BIN files were **26.10% smaller** than FP16,
but the measured English/Tamil clips showed **no latency speedup**. Decoder
execution used f32 arithmetic, not all-INT8 compute. See [README](README.md)
for timing details, scope, sources and licenses.
Fixing conversion does not fix the original model's high Tamil WER.

**Refreshed local analysis:** on Core Ultra X7 358H / Arc B390, four-language
complete-generation measurements again showed no mixed-weight GPU speedup.
Reference-named kernels were outside the top five (2.329% FP16 / 2.236% mixed
aggregate kernel time); f32 decode `lm_head` was the largest node. All 28
gate-projection/Swish/Multiply patterns were already fused. Shared-input Q/K/V
projections and separate attention Gemm/SoftMax paths are future opportunities,
not implemented optimizations. The requested PTL roofline estimates f32
prefill as compute-bound and first decode as memory-bound, with retained
decoder shape warnings; estimates are not hard hardware-efficiency bounds.
Validated phase replay completed prefill then decode before resetting and
checked actual-generation top-1 plus repeat-logit stability. See the
[README findings](README.md#local-performance-and-profiling-findings) for
the five-runtime timing table, protocol and limitations. These static findings
do not add research tools to the notebook's runtime dependencies.

Conversion, compression, inference and measurement are demonstrated; multilingual
deployment is **not** approved. No PyTorch XPU, activation PTQ, strict raw-logit
numerical certificate, speaker-disjoint validation, or broad hardware certification.
Profiling/roofline/recovery investigations remain outside this tutorial's runtime
dependencies. After shutting down the kernel, generated `model/`, `data/`, `output/`
may be removed; do not delete shared model caches or other workspaces.